In [1]:
!pip install -q transformers peft trl accelerate bitsandbytes datasets sentencepiece safetensors

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 518.9/518.9 kB 11.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.1/59.1 MB 16.7 MB/s eta 0:00:00


In [2]:

from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    Trainer,
    TrainingArguments,
    DataCollatorForLanguageModeling
)

from peft import (
    LoraConfig,
    get_peft_model,
    PeftModel,
    prepare_model_for_kbit_training
)

from datasets import load_dataset
import torch

print("All imports successful")



All imports successful


In [3]:

model_name = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

tokenizer = AutoTokenizer.from_pretrained(
    model_name,
    trust_remote_code=True,
    padding_side="right"
)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    load_in_4bit=True,
    device_map="auto",
    trust_remote_code=True,
    torch_dtype=torch.float16
)

model = prepare_model_for_kbit_training(model)
model.config.use_cache = False
model.gradient_checkpointing_disable()

print("Model loaded in 4-bit mode")




/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/551 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/608 [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!
The `load_in_4bit` and `load_in_8bit` arguments are deprecated and will be removed in the future versions. Please, pass a `BitsAndBytesConfig` object in `quantization_config` argument instead.


model.safetensors:   0%|          | 0.00/2.20G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

Model loaded in 4-bit mode


In [5]:
data_files = {
    "train": "/content/train.jsonl",
    "validation": "/content/val.jsonl"
}

dataset = load_dataset("json", data_files=data_files)

train_data = dataset["train"]
val_data = dataset["validation"]


Generating train split: 0 examples [00:00, ? examples/s]

Generating validation split: 0 examples [00:00, ? examples/s]

In [6]:
def format_example(example):
    instruction = example["instruction"]
    input_text = example.get("input", "")
    output = example["output"]

    if input_text.strip():
        text = (
            "### Instruction:\n" + instruction +
            "\n\n### Input:\n" + input_text +
            "\n\n### Response:\n" + output
        )
    else:
        text = (
            "### Instruction:\n" + instruction +
            "\n\n### Response:\n" + output
        )

    return {"text": text}

train_data = train_data.map(format_example)
val_data = val_data.map(format_example)



Map:   0%|          | 0/900 [00:00<?, ? examples/s]

Map:   0%|          | 0/100 [00:00<?, ? examples/s]

In [7]:

def tokenize_function(examples):
    return tokenizer(
        examples["text"],
        truncation=True,
        max_length=512,
        padding="max_length"
    )

train_data = train_data.map(
    tokenize_function,
    batched=True,
    remove_columns=train_data.column_names
)

val_data = val_data.map(
    tokenize_function,
    batched=True,
    remove_columns=val_data.column_names
)

print("Dataset tokenized")



Map:   0%|          | 0/900 [00:00<?, ? examples/s]

Map:   0%|          | 0/100 [00:00<?, ? examples/s]

Dataset tokenized


In [8]:
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()


trainable params: 4,505,600 || all params: 1,104,553,984 || trainable%: 0.4079


In [9]:
training_args = TrainingArguments(
    output_dir="./lora_output",
    num_train_epochs=3,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    fp16=True,
    logging_steps=10,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=1,
    optim="paged_adamw_8bit",
    report_to="none"
)


In [10]:

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_data,
    eval_dataset=val_data,
    data_collator=DataCollatorForLanguageModeling(
        tokenizer=tokenizer,
        mlm=False
    )
)



In [11]:

trainer.train()



Epoch,Training Loss,Validation Loss
1,0.109400,0.114184
2,0.104900,0.107901
3,0.106700,0.106140


TrainOutput(global_step=675, training_loss=0.13979410392266733, metrics={'train_runtime': 941.7757, 'train_samples_per_second': 2.867, 'train_steps_per_second': 0.717, 'total_flos': 8618030766489600.0, 'train_loss': 0.13979410392266733, 'epoch': 3.0})

In [13]:
output_dir = "./lora_output/final"

model.save_pretrained(output_dir)
tokenizer.save_pretrained(output_dir)

print("LoRA adapter saved")


LoRA adapter saved


In [15]:
base_model = AutoModelForCausalLM.from_pretrained(
    model_name,
    load_in_4bit=True,
    device_map="auto",
    trust_remote_code=True,
    torch_dtype=torch.float16
)

inference_model = PeftModel.from_pretrained(base_model, output_dir)
inference_model.eval()

prompt = (
    "### Instruction:\n"
    "What is Diabetes ? \n\n"
    "### Response:\n"
)

inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

with torch.no_grad():
    outputs = inference_model.generate(
        **inputs,
        max_new_tokens=120,
        temperature=0.7,
        top_p=0.9,
        do_sample=True
    )

print(tokenizer.decode(outputs[0], skip_special_tokens=True))



The `load_in_4bit` and `load_in_8bit` arguments are deprecated and will be removed in the future versions. Please, pass a `BitsAndBytesConfig` object in `quantization_config` argument instead.


### Instruction:
What is Diabetes ? 

### Response:
Diabetes is a medical condition that affects the body and requires clinical management. It is often met with weight loss, frequent urination, and fatigue. The primary symptoms are shortness of breath, frequent urination, weight loss, fever, persistent cough, joint pain, weight gain, fatigue, chest pain, frequent urination, shortness of breath, weight loss, fever, persistent cough, joint pain, weight gain, fatigue, chest pain, frequent urination, shortness of breath, weight loss, fever, persistent c
